In [46]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_curve
from scipy.sparse import hstack

import lightgbm as lgb

In [29]:
user = pd.read_csv("C:/Users/laoti/Desktop/Grad School/BUDT758T/Data&Sample/user_clean.csv")
business = pd.read_csv("C:/Users/laoti/Desktop/Grad School/BUDT758T/Data&Sample/business_data_cleaned.csv")
tip = pd.read_csv("C:/Users/laoti/Desktop/Grad School/BUDT758T/Data&Sample/tips_cleaned.csv")

train_x = pd.read_csv("C:/Users/laoti/Desktop/Grad School/BUDT758T/Data&Sample/review_train_x.csv")
train_y = pd.read_csv("C:/Users/laoti/Desktop/Grad School/BUDT758T/Data&Sample/review_train_y.csv")
test_x = pd.read_csv("C:/Users/laoti/Desktop/Grad School/BUDT758T/Data&Sample/review_test_x.csv")

In [30]:
user = user.rename(columns={
    "review_count": "user_review_count",
    "fans": "user_fans",
    "average_stars": "user_avg_stars",
    "friends_count": "user_friends_count",
    "is_elite": "user_is_elite",
    "elite_years": "user_elite_years"
})

In [31]:
business = business.rename(columns={
    "review_count": "business_review_count",
    "stars": "business_stars",
    "is_open": "business_is_open"
})

In [32]:
tip = tip.rename(columns={
    "compliment_count": "tip_compliment"
})

In [33]:
train = pd.concat([train_x, train_y], axis=1)

In [34]:
train = train.merge(user, on="user_id", how="left")
test_x = test_x.merge(user, on="user_id", how="left")

train = train.merge(business, on="business_id", how="left")
test_x = test_x.merge(business, on="business_id", how="left")

In [35]:
tip_agg = tip.groupby("business_id").agg({
    "tip_compliment": "sum"
}).reset_index()

tip_agg = tip_agg.rename(columns={
    "tip_compliment": "tip_compliments"
})

train = train.merge(tip_agg, on="business_id", how="left")
test_x = test_x.merge(tip_agg, on="business_id", how="left")

In [36]:
user_features = [
    "user_review_count",
    "user_fans",
    "user_avg_stars",
    "user_friends_count",
    "user_is_elite",
    "user_elite_years"
]

business_features = [
    "business_stars",
    "business_review_count",
    "business_is_open"
]

all_struct_features = user_features + business_features + ["tip_compliments"]

In [37]:
print(train.columns)

Index(['review_id', 'user_id', 'business_id', 'stars', 'text', 'date',
       'top_useful', 'user_review_count', 'yelping_since', 'useful',
       ...
       'Entertainment', 'Travel', 'Shopping', 'HealthandFitness', 'Automotive',
       'Pets', 'Activelife', 'Other', 'is_hours_missing', 'tip_compliments'],
      dtype='object', length=131)


缺失

In [38]:
for col in all_struct_features:
    train[col] = train[col].fillna(0)
    test_x[col] = test_x[col].fillna(0)

文本特征

In [39]:
train["text"] = train["text"].fillna("")
test_x["text"] = test_x["text"].fillna("")

tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=5,
    stop_words="english"
)

X_text = tfidf.fit_transform(train["text"])
X_test_text = tfidf.transform(test_x["text"])

In [40]:
train["text_len"] = train["text"].apply(len)
test_x["text_len"] = test_x["text"].apply(len)

all_struct_features.append("text_len")

拼特征

In [41]:
X_struct = train[all_struct_features].values
X_test_struct = test_x[all_struct_features].values

In [42]:
X_all = hstack([X_text, X_struct])
X_test_all = hstack([X_test_text, X_test_struct])

y = train["top_useful"]

In [50]:
X_tr, X_val, y_tr, y_val, idx_tr, idx_val = train_test_split(
    X_all, y, train.index, test_size=0.2, random_state=42
)

In [47]:
model_text = SGDClassifier(
    loss="log_loss",
    max_iter=1000,
    alpha=1e-4,
    class_weight="balanced"
)

model_text.fit(X_tr, y_tr)

In [51]:
X_lgb_tr = train.loc[idx_tr, all_struct_features]
X_lgb_val = train.loc[idx_val, all_struct_features]

In [54]:
X_lgb = train[all_struct_features]
X_test_lgb = test_x[all_struct_features]

model_lgb = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_lgb.fit(X_lgb_tr, y_tr)
val_probs_lgb = model_lgb.predict_proba(X_lgb_val)[:,1]

[LightGBM] [Info] Number of positive: 152873, number of negative: 1108949
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060653 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1536
[LightGBM] [Info] Number of data points in the train set: 1261822, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.121153 -> initscore=-1.981560
[LightGBM] [Info] Start training from score -1.981560


In [55]:
print(val_probs_text.shape)
print(val_probs_lgb.shape)

(315456,)
(315456,)


In [56]:
val_probs = 0.7 * val_probs_text + 0.3 * val_probs_lgb

In [57]:
fpr, tpr, thresholds = roc_curve(y_val, val_probs)

best_idx = np.argmax([tpr[i] if fpr[i] <= 0.10 else -1 for i in range(len(fpr))])
best_threshold = thresholds[best_idx]

print("FPR:", fpr[best_idx])
print("TPR:", tpr[best_idx])

FPR: 0.09998123976304378
TPR: 0.4888436014004285


In [59]:
test_probs_text = model_text.predict_proba(X_test_all)[:,1]
test_probs_lgb  = model_lgb.predict_proba(X_test_lgb)[:,1]

final_test_probs = 0.7 * test_probs_text + 0.3 * test_probs_lgb

preds = (final_test_probs >= best_threshold).astype(int)

In [60]:
print(len(preds))        # 700000
print(np.unique(preds))  # [0,1]
print(pd.isna(preds).sum())

700000
[0 1]
0


In [61]:
pd.Series(preds).to_csv(
    "group_7_yelp.csv",
    index=False,
    header=False
)